# GIAI ĐOẠN 3.0.3: ĐÁNH GIÁ TẬP KIỂM THỬ ĐỘC LẬP (OUT-OF-SAMPLE TEST SET EVALUATION)

**Môn học:** IS403 - Khai phá dữ liệu và ứng dụng trong tài chính
**Đề tài:** Đánh giá khả năng dự đoán lợi suất cổ phiếu thị trường chứng khoán Mỹ (S&P 500) qua các khung thời gian và trạng thái biến động thị trường.
**Tập dữ liệu kiểm định:** **Split 3** (02/11/2020 – 07/04/2026, 1,362 phiên giao dịch thực tế).

---

### Nguyên tắc kiểm định nghiêm ngặt (Zero-Data-Leakage Protocol):
1. **Tập dữ liệu độc lập (Strictly Sealed):** Tập Test (Split 3) hoàn toàn tách biệt, chưa từng được tiếp cận trong quá trình huấn luyện và chọn siêu tham số (Splits 1 & 2).
2. **Marcos López de Prado Purging:** Áp dụng Purging $h-1$ ngày tại cuối tập Train để loại bỏ 100% rủi ro rò rỉ nhãn lợi suất gối đầu vào tập Test.
3. **Chuẩn hóa độc lập:** `StandardScaler` chỉ được `.fit()` trên tập huấn luyện đã Purged, sau đó `.transform()` tập Test.
4. **Benchmark học thuật:** Campbell & Thompson (2008) Historical Mean ($y_{bench} = \text{mean}(y_{train})$).
5. **Kiểm định thống kê:** Diebold-Mariano test (hiệu chỉnh mẫu hữu hạn HLN 1997) và Binomial Sign test cho độ chính xác phương hướng (DA).

## 1. Khởi tạo môi trường & Cấu hình đường dẫn

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập hiển thị đẹp
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 25)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Import module testing chuyên biệt
import test_config as config
import test_engine as engine
import test_analysis as analysis

print(f"Device khả dụng: {config.DEVICE}")
print(f"Tập dữ liệu: {config.DATA_PATH}")
print(f"Số lượng mô hình kiểm định: {len(config.MODELS)}")
print(f"Khung thời gian (Horizons): {config.HORIZONS}")

## 2. Kiểm tra tập dữ liệu & Kiểm soát rò rỉ thông tin (Leakage Audit)

In [ ]:
df = engine.load_dataset()
train_df = df[df['Split'].isin([1, 2])].copy()
test_df = df[df['Split'] == 3].copy()

print("=== BÁO CÁO PHÂN CHIA DỮ LIỆU ===")
print(f"Train Set (Splits 1 + 2): {len(train_df):,} dòng | {train_df['Date'].iloc[0].date()} -> {train_df['Date'].iloc[-1].date()}")
print(f"Test Set (Split 3):       {len(test_df):,} dòng | {test_df['Date'].iloc[0].date()} -> {test_df['Date'].iloc[-1].date()}")
print("\nPhân bố Volatility Regime trên tập Test (Split 3):")
print(test_df['Volatility_Regime'].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))

## 3. Tải bộ siêu tham số tối ưu (Fold 5 Tuned Parameters)

In [ ]:
best_params = config.load_best_hyperparameters()
tune_summary = []
for (mod, hrz), p in best_params.items():
    tune_summary.append({'Model': mod, 'Horizon': hrz, 'Best_Params': str(p)})
df_tune_view = pd.DataFrame(tune_summary)
print(f"Đã tải {len(df_tune_view)} cấu hình tham số tối ưu từ Lan01 Walk-Forward Tuning:")
df_tune_view.head(15)

## 4. Thực thi kiểm định Out-of-Sample trên Split 3
> **Lưu ý:** Bạn có thể chạy toàn bộ 14 mô hình hoặc lọc các mô hình mong muốn bằng tham số `models`.

In [ ]:
# Nếu đã có file dự đoán lưu sẵn, tải trực tiếp; nếu chưa có, tiến hành huấn luyện & dự báo
if os.path.exists(config.TEST_PREDICTIONS_PATH):
    print(f"Đã tìm thấy dự đoán có sẵn tại: {config.TEST_PREDICTIONS_PATH}")
    preds_df = pd.read_csv(config.TEST_PREDICTIONS_PATH)
else:
    print("Tiến hành huấn luyện và dự báo trên toàn bộ Split 3...")
    preds_df = engine.run_test_evaluation(models=config.MODELS, horizons=config.HORIZONS)

print(f"Tổng số mẫu dự báo trên Test Set: {len(preds_df):,}")
preds_df.head()

## 5. Tính toán chỉ số hiệu năng hồi quy & Kiểm định thống kê

In [ ]:
df_reg, df_cls, df_stat, df_regime = analysis.compute_metrics_and_tests(preds_df)

print("=== BẢNG CHỈ SỐ HỒI QUY TRÊN TEST SET (OOS R2, MAE, RMSE, DA) ===")
display(df_reg)

print("\n=== BẢNG KIỂM ĐỊNH THỐNG KÊ (DIEBOLD-MARIANO & SIGN TEST) ===")
display(df_stat)

## 6. Chỉ số mô hình phân loại (Logistic Regression)

In [ ]:
print("=== HIỆU NĂNG PHÂN LOẠI XU HƯỚNG CỦA LOGISTIC REGRESSION ===")
display(df_cls)

## 7. Phân tích theo chế độ biến động (Volatility Regime Analysis)

In [ ]:
print("=== HIỆU NĂNG THEO CHẾ ĐỘ BIẾN ĐỘNG TRONG GIAI ĐOẠN 2020-2026 ===")
display(df_regime.head(20))

## 8. Trực quan hóa kết quả (Visualizations)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Xu hướng OOS R2 qua các khung thời gian
if not df_reg.empty:
    sns.barplot(data=df_reg, x='Horizon', y='OOS_R2', hue='Model', ax=axes[0])
    axes[0].set_title("Campbell-Thompson OOS R² across Horizons (Split 3: 2020-2026)", fontsize=12, fontweight='bold')
    axes[0].set_ylabel("OOS R²")
    axes[0].axhline(0, color='red', linestyle='--', linewidth=1, label='Historical Mean Benchmark')
    axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# 2. Độ chính xác phương hướng (Directional Accuracy)
if not df_reg.empty:
    sns.barplot(data=df_reg, x='Horizon', y='DA', hue='Model', ax=axes[1])
    axes[1].set_title("Directional Accuracy (DA) across Horizons", fontsize=12, fontweight='bold')
    axes[1].set_ylabel("Directional Accuracy (0 to 1)")
    axes[1].axhline(0.5, color='black', linestyle=':', linewidth=1.5, label='Random Guess (50%)')
    axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

## 9. Tự động xuất báo cáo học thuật (Markdown Report)

In [ ]:
report_md = analysis.generate_markdown_report(df_reg, df_cls, df_stat, df_regime)
print(f"Đã xuất báo cáo tổng hợp Markdown thành công tại: {config.TEST_REPORT_PATH}")